# Using Apptainer in JupyDo

[Apptainer](https://apptainer.org) (formerly Singularity) runs **containers inside your environment**, as a
normal user, without root: useful to run tools published as Docker or Apptainer images (for example the
thousands of bioinformatics tools of [BioContainers](https://biocontainers.pro)) without installing them.

**Requirements**
- the environment must be created with the **Apptainer** option checked (*Optional modules* in the
  environment form). Existing environments get it only if you create a new one (or Reset this one and
  choose it again);
- the image of the environment must be of the **same architecture as the server** (e.g. an amd64 image on an
  amd64 server): Apptainer cannot run in an emulated environment, and JupyDo tells you so when it starts.
  From a normal environment you can still run containers of the other architecture (section 6).

The commands below are shell commands: in a notebook they are in `%%bash` cells; you can also type them in a
terminal (*File → New → Terminal*).

## 1. Is Apptainer here?

In [ ]:
%%bash
apptainer --version

If you get `apptainer: command not found`, this environment was created without the Apptainer option.

## 2. Where to keep images and cache
Apptainer keeps a **cache** of what it downloads, and you will keep the images you use (`.sif` files).
Put both in `/sharedFolder`: it is your persistent storage. Your home folder lives inside the environment
and is **lost when the environment is Reset**.

This sets the cache folder for every command run from this notebook:

In [ ]:
%env APPTAINER_CACHEDIR=/sharedFolder/apptainer/cache
import os
for d in ("cache", "images", "data", "results"):
    os.makedirs(f"/sharedFolder/apptainer/{d}", exist_ok=True)
print("ready: /sharedFolder/apptainer/{cache,images,data,results}")

(In a terminal: `export APPTAINER_CACHEDIR=/sharedFolder/apptainer/cache`, or add that line to
`~/.bashrc`.)

## 3. Run a container from Docker Hub
`apptainer exec <image> <command>` runs a command inside a container. `docker://...` takes the image from
Docker Hub (or another registry): the first time it is downloaded and converted, then it comes from the cache.

In [ ]:
%%bash
apptainer exec docker://alpine:3.20 cat /etc/alpine-release

## 4. Keep an image as a `.sif` file
`apptainer pull` saves the image as a single `.sif` file: you can reuse it, share it, and it never changes
(good for reproducibility: the same file gives the same tool, also in years).

In [ ]:
%%bash
cd /sharedFolder/apptainer/images
[ -f alpine_3.20.sif ] || apptainer pull alpine_3.20.sif docker://alpine:3.20
ls -lh alpine_3.20.sif
apptainer exec alpine_3.20.sif cat /etc/alpine-release

## 5. A bioinformatics tool: samtools from BioContainers
We count the reads of a tiny SAM file with `samtools`, without installing samtools.

Inside the container Apptainer automatically shows your home folder, `/tmp` and the current folder. Other
folders must be listed with `--bind` (`-B`): here `/sharedFolder`, where the data are.

In [ ]:
%%bash
cd /sharedFolder/apptainer
# a tiny SAM file with two reads (tab separated)
printf '@HD\tVN:1.6\tSO:unsorted\n@SQ\tSN:chr1\tLN:1000\n' > data/test.sam
printf 'read1\t0\tchr1\t100\t60\t10M\t*\t0\t0\tACGTACGTAC\tIIIIIIIIII\n' >> data/test.sam
printf 'read2\t16\tchr1\t200\t60\t10M\t*\t0\t0\tTTTTGGGGCC\tIIIIIIIIII\n' >> data/test.sam

[ -f images/samtools_1.9.sif ] || apptainer pull images/samtools_1.9.sif docker://biocontainers/samtools:v1.9-4-deb_cv1
apptainer exec --bind /sharedFolder images/samtools_1.9.sif samtools --version | head -1
echo "reads in test.sam:"
apptainer exec --bind /sharedFolder images/samtools_1.9.sif samtools view -c /sharedFolder/apptainer/data/test.sam

It prints `2`. Any image of BioContainers (`quay.io/biocontainers/<tool>:<version>`) or Docker Hub is used the
same way: pick the exact version tag on the tool's page, so the analysis can be repeated with the same version.

## 6. Containers of the other architecture
From a normal environment you can also run containers built for the **other architecture** (e.g. arm64 on an
amd64 server): their programs are emulated by the server, so they are **much slower**.
For these use a **sandbox folder**, not a `.sif` file (with a `.sif` Apptainer refuses them inside JupyDo):

In [ ]:
%%bash
cd /sharedFolder/apptainer/images
[ -d alpine-arm64 ] || apptainer build --arch arm64 --fix-perms --sandbox alpine-arm64 docker://alpine:3.20
apptainer exec alpine-arm64 uname -m

It prints `aarch64` (arm64). On an arm64 server the same works the other way round with `--arch amd64`.

## 7. Apptainer in the job queue
Jobs (see the job queue examples) run in a copy of this environment, with Apptainer working as here.
Heavy containerised tools belong in a job: here the samtools count of section 5 as a job, writing its result
to a file.

In [ ]:
%%job --time 10m --cpus 1 --mem 2 --name apptainer-samtools
import os, subprocess
env = dict(os.environ, APPTAINER_CACHEDIR="/sharedFolder/apptainer/cache")
result = subprocess.run(
    ["apptainer", "exec", "--bind", "/sharedFolder",
     "/sharedFolder/apptainer/images/samtools_1.9.sif",
     "samtools", "view", "-c", "/sharedFolder/apptainer/data/test.sam"],
    capture_output=True, text=True, env=env, check=True)
with open("/sharedFolder/apptainer/results/read_count.txt", "w") as f:
    f.write(result.stdout)
print("reads:", result.stdout.strip())

In [ ]:
# when %jobs says the job is 'done':
print(open("/sharedFolder/apptainer/results/read_count.txt").read())

## 8. GPUs: Apptainer in a GPU job
GPUs are **only available to jobs** started with `--gpus N` (see `jobqueue_gpu.ipynb`), not to your
interactive environment. Inside such a job, `apptainer exec --nv` gives the GPU to the container: Apptainer
brings in the server's NVIDIA driver, so **any image** can use the GPU; the image only needs the CUDA
libraries its own programs use.

First, download the images **here** (no GPU needed to download; a job should not spend its time on it):
- a plain Ubuntu, to see the GPU with `nvidia-smi`;
- the official PyTorch image (CUDA included), for a real computation. It is **several GB**: download it once.

In [ ]:
%%bash
cd /sharedFolder/apptainer/images
[ -f ubuntu_24.04.sif ] || apptainer pull ubuntu_24.04.sif docker://ubuntu:24.04
[ -f pytorch_2.4.1_cuda12.1.sif ] || apptainer pull pytorch_2.4.1_cuda12.1.sif docker://pytorch/pytorch:2.4.1-cuda12.1-cudnn9-runtime
ls -lh ubuntu_24.04.sif pytorch_2.4.1_cuda12.1.sif

**Which GPU did the job get?** `nvidia-smi` runs inside the Ubuntu container, with the server's driver
brought in by `--nv`:

In [ ]:
%%job --time 5m --cpus 1 --mem 2 --gpus 1 --name apptainer-gpu-check
import os, subprocess
env = dict(os.environ, APPTAINER_CACHEDIR="/sharedFolder/apptainer/cache")
out = subprocess.run(["apptainer", "exec", "--nv", "/sharedFolder/apptainer/images/ubuntu_24.04.sif",
                      "nvidia-smi", "-L"], capture_output=True, text=True, env=env, check=True).stdout
with open("/sharedFolder/apptainer/results/gpu_check.txt", "w") as f:
    f.write(out)
print(out)

**A real computation:** PyTorch, from its official container, multiplies two large matrices on the GPU and
saves the timing. Nothing is installed in your environment: everything is in the `.sif` file.

In [ ]:
%%job --time 15m --cpus 2 --mem 8 --gpus 1 --name apptainer-gpu-torch
import os, subprocess
env = dict(os.environ, APPTAINER_CACHEDIR="/sharedFolder/apptainer/cache")
program = '''
import json, time, torch
assert torch.cuda.is_available(), "no GPU visible in the container"
x = torch.rand(8000, 8000, device="cuda")
torch.cuda.synchronize(); start = time.time()
for _ in range(10):
    y = x @ x
torch.cuda.synchronize()
result = {"gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
          "seconds_for_10_matmul_8000": round(time.time() - start, 3)}
json.dump(result, open("/sharedFolder/apptainer/results/gpu_torch.json", "w"))
print(result)
'''
out = subprocess.run(["apptainer", "exec", "--nv", "--bind", "/sharedFolder",
                      "/sharedFolder/apptainer/images/pytorch_2.4.1_cuda12.1.sif", "python", "-c", program],
                     capture_output=True, text=True, env=env)
print(out.stdout, out.stderr)
out.check_returncode()

In [ ]:
# when %jobs says both jobs are 'done':
print(open("/sharedFolder/apptainer/results/gpu_check.txt").read())
print(open("/sharedFolder/apptainer/results/gpu_torch.json").read())

## 9. Common problems
| Message | What to do |
|---|---|
| `apptainer: command not found` | the environment was created without the Apptainer option: create one with it |
| `the image's architecture (...) could not run on the host's (...)` | an image of the other architecture as `.sif`: use a sandbox folder (section 6) |
| `No space left on device` | the cache is big: `apptainer cache clean` (keep your `.sif` files) |
| a file is not found inside the container | its folder is not shown in the container: add `--bind /that/folder` |
| `WARNING: Could not find any nv files on this host!` | no GPU here: GPUs are only in jobs started with `--gpus` (section 8) |